In [2]:
from BCP303.BCP303 import BPC303
from Sourcemeter.sourcemeter import Sourcemeter2401
import time
from datetime import datetime
from tool.tools import post_process


In [ ]:
# move the stage in Z direction
bcp303_z = BPC303(channel_id=1)
position_z = bcp303_z.move_to_origin(start_position=0)
step_size_z = 0.1  # move 0.1 mm
position_z = bcp303_z.bcp303_move_stage(step_size=step_size_z, current_position=position_z,)
bcp303_z.bcp303_stop(ifback=True)

In [ ]:
# move the stage in x direction
bcp303_x = BPC303(channel_id=2)
position_x = bcp303_x.move_to_origin(start_position=0)
step_size_x = 0.1  # move 0.1 mm
position_x = bcp303_x.bcp303_move_stage(step_size=step_size_x, current_position=position_x,)
bcp303_x.bcp303_stop(ifback=False)

In [ ]:
# record the signals
sm2401 = Sourcemeter2401(speed_nplc=0.1)


In [ ]:
# sweep operation: move stage in z-direction and record the signals

def sweep_operation(stage_settings=None,chip_name="chip_test",sample_name="beam_test"):
    step_number = stage_settings["step_number"]
    time_interval = stage_settings["time_interval"]
    position_x = stage_settings["position_x"]
    step_size_z = stage_settings["step_size_z"]
    count = 1
    formatted_time = datetime.now().strftime("%Y%m%d%H%M")
    try:
        bcp303_z = BPC303(channel_id=1)
        bcp303_device = bcp303_z.get_device()
        # moving controller x
        bcp303_x = BPC303(channel_id=2, device=bcp303_device)
        # Sourcemeter
        sm2401 = Sourcemeter2401(speed_nplc=0.1)
        # set the initial position of the stage z
        position_z = bcp303_z.move_to_origin(start_position=stage_settings["position_z"]) - step_size_z
        time.sleep(1)
        bcp303_x.move_to_origin()
        time.sleep(1)
        allData = [{"position": [], "voltage": []}]
        for _ in range(step_number):
            position_z = bcp303_z.bcp303_move_stage(
                step_size=step_size_z,
                current_position=position_z,
            )
            allData[0]["position"].append(position_z)
            time.sleep(0.5)
            bcp303_x.move_to_position(start_position=0, step_size=0.5, final_position=position_x)
            time.sleep(0.5)
            step_start = time.perf_counter()
            voltage = sm2401.measure_voltage(duration=time_interval / 2, dt=0.01)[
                "voltage"
            ]
            allData[0]["voltage"].append(voltage)
            # remaining time to wait until the next step
            elapsed = time.perf_counter() - step_start
            remaining = time_interval - elapsed
            if remaining > 0:
                time.sleep(remaining)
            bcp303_x.move_to_origin()
            time.sleep(0.5)
            print(f"process completed: {100 * count / step_number:.1f}%")
            count += 1
        sm2401_settings = sm2401.getSettings()
        stage_settings["position_x"] = position_x
        settings = {"stage": stage_settings, "sourcemeter": sm2401_settings}
        post_process(
            chip_name=chip_name,
            sample_name=sample_name,
            result=allData[0],
            config=settings,
            repeat=1,
            position_z=position_z,
            ifshow=False,
            formatted_time=formatted_time,
        )
    # stop the stage and sourcemeter
    except Exception as e:
        print(f"Exception occurred: {e}")
    finally:
        time.sleep(0.5)
        sm2401.close()
        bcp303_z.move_to_origin()
        bcp303_z.channel.StopPolling()
        bcp303_x.bcp303_stop(ifback=True)
        return allData[0], settings
    

setting = {
        "position_x": 3.5,
        "step_number": 15,
        "position_z": 0,
        "step_size_z": 1,
        "time_interval": 2,  
    }
sweep_operation(
        stage_settings=setting,
        chip_name="V1_R_W_2_Right",
        # chip_name="SiN_beam",
        # sample_name="w2_sweep",  # test_1_right, w=20
        sample_name="w5_3_sweep",  
    )

APT High Voltage Amplifier initialized
APT High Voltage Amplifier initialized
IDN: KEITHLEY INSTRUMENTS INC.,MODEL 2401,4614233,B02 Jan 20 2021 10:19:49/B01  /W/N
process completed: 6.7%
process completed: 13.3%
process completed: 20.0%
process completed: 26.7%
process completed: 33.3%
process completed: 40.0%
process completed: 46.7%
process completed: 53.3%
process completed: 60.0%
process completed: 66.7%
process completed: 73.3%
process completed: 80.0%
process completed: 86.7%
process completed: 93.3%
process completed: 100.0%
Config saved to: ./result/SiN_beam/202606150905_w2_sweep\202606150905_SiN_beam_w2_sweep_config.json
Stage Moving Done
Stage disconnected


({'position': [0.0,
   0.995513779107028,
   1.98980681783502,
   2.98409985656301,
   3.97778252510147,
   4.96902371288186,
   5.96636860255745,
   6.9533371990112,
   7.94946134830775,
   8.9455854976043,
   9.94537186803796,
   10.9445478682821,
   11.9443342387158,
   12.944730979339,
   13.9457380901517],
  'voltage': [[0.0208897,
    0.02145974,
    0.02097113,
    0.02114272,
    0.02178696,
    0.02104038,
    0.0213565,
    0.02170471,
    0.02153295,
    0.02128518,
    0.02130817,
    0.02110645,
    0.02138561,
    0.02101317,
    0.02149088,
    0.02140985],
   [0.02009193,
    0.02082647,
    0.020966,
    0.02075721,
    0.02094392,
    0.02099511,
    0.02087267,
    0.02105231,
    0.02078928,
    0.02095604,
    0.02107838,
    0.0209168,
    0.02086978,
    0.02073411,
    0.02078932,
    0.02110669],
   [0.02030286,
    0.0202051,
    0.02040085,
    0.02016209,
    0.02016324,
    0.02024045,
    0.02023943,
    0.01975457,
    0.01995447,
    0.02036106,
    0.02